# 0. Base Normalizer

In [1]:
import re

class BaseNormalizer:
    
    in_regex = None
    out_regex = None
    
    def normalize(self, text):
        return re.sub(self.in_regex, self.out_regex, text)
    
    def __call__(self, text, *args, **kwds):
        return self.normalize(text)

In [2]:
def sort_by_key(dictionary: dict):
    return dict(sorted(dictionary))

In [3]:
import json

def load_json(filepath, is_sort_by_key=True):
    
    with open(filepath, "r", encoding="utf-8") as file:
        data = json.load(file)
        
        if isinstance(data, dict):
            text_bank = sort_by_key(data.items()) if is_sort_by_key else data
        
        elif isinstance(data, list):
            text_bank = data
        
        else:
            raise TypeError("Input data must be either a dictionary or a list")
    
    return text_bank

In [4]:
import re

class BaseBankNormalizer(BaseNormalizer):

    bank_path = None
    bank = None
    
    def normalize(self, text):
        
        if self.bank is None:
            self.bank = load_json(self.bank_path)
        
        if self.in_regex is None:
            self.in_regex = r"\b(" + '|'.join([re.escape(k) for k in self.bank.keys()]) + r")\b"
            self.out_regex = lambda match: self.bank[match.group(0)]
            
        return super().normalize(text)

# 1. Whitespace Normalizer

In [5]:
import re

class WhitespaceNormalizer(BaseNormalizer):
    
    in_regex = r'\s+'
    out_regex = r' '
    
    def normalize(self, text):
        text = text.strip()
        return super().normalize(text)

In [6]:
whitespace_normalizer = WhitespaceNormalizer()
text = "  Đây   là   một  ví   dụ   với   tiếng   Việt   .   \n\t"
normalized_text = whitespace_normalizer(text)
normalized_text

'Đây là một ví dụ với tiếng Việt .'

# 2. Acronym Normalizer

In [7]:
import re

class AcronymNormalizer(BaseBankNormalizer):
    bank_path = r'text_banks/acronyms.json'


In [8]:
acronym_normalizer = AcronymNormalizer()
text = "anqp"
normalized_text = acronym_normalizer(text)
normalized_text

'an ninh quốc phòng'

# 3. Date normalizer

In [41]:
import re

class DateNormalizer(BaseNormalizer):
    in_regex = r"(ngày|tháng)?\s*((1[0-2]|0?[1-9])[-/](\d{4})|([12][0-9]|3[01]|0?[1-9])[-/](1[0-2]|0?[1-9])([-/]\d{4})?)"
    
    def out_regex_func(self, match):
        prefix = match.group(1)
        day = match.group(5)
        month = match.group(3) if match.group(3) else match.group(6)
        year = match.group(4) if match.group(4) else match.group(7)       

        if prefix == "tháng" or day is None:
            return f"tháng {month} năm {year}"
        
        if prefix is not None and prefix != "ngày":
            result = f"{prefix} ngày {day} tháng {month}"
    
        else:
            result = f"ngày {day} tháng {month}"
        
        if year:
            result += f" năm {year[1:]}"
        
        return result

    def normalize(self, text):
        
        if self.out_regex is None:
            self.out_regex = self.out_regex_func
            
        return super(DateNormalizer, self).normalize(text)

In [42]:
date_normalizer = DateNormalizer()

In [51]:
date1 = "ngày 15/10/2024,"
date_normalizer(date1)

'ngày 15 tháng 10 năm 2024,'

In [44]:
date2 = "11/12/2002"
date_normalizer(date2)

'ngày 11 tháng 12 năm 2002'

In [45]:
date3 = "11/12"
date_normalizer(date3)

'ngày 11 tháng 12'

In [46]:
date4 = "tháng 11/2002"
date_normalizer(date4)

'tháng 11 năm 2002'

# 4. Letter Normalizer

In [15]:
class LetterNormalizer(BaseBankNormalizer):
    bank_path = r'text_banks/letters.json'
    bank = None
    
    def normalize(self, text):
        
        if self.bank is None:
            self.bank = load_json(self.bank_path)
        
        if self.in_regex is None:
            self.in_regex = r"\b(" + '|'.join([re.escape(k) for k in self.bank.keys()]) + r")(\d{0,})?\b"
            self.out_regex = lambda match: self.bank[match.group(1)] + (" " + match.group(2)) if match.group(2) else ""
            
        return super().normalize(text)

In [16]:
letter_normalizer = LetterNormalizer()
text = "b123"
letter_normalizer(text)

'bê 123'

# 5. Symbol Normalizer

In [17]:
class SymbolNormalizer(BaseBankNormalizer):
    bank_path = r'text_banks/symbols.json'

In [18]:
symbol_normalizer = SymbolNormalizer()
symbol = "χ"
symbol_normalizer(symbol)

'chi'

# 6. Same Phoneme Normalizer

In [19]:
class SamePhonemeNormalizer(BaseBankNormalizer):
    bank_path = r'text_banks/same_phonemes.json'

In [20]:
same_phoneme_normalizer = SamePhonemeNormalizer()
same_phoneme = "nghe"
same_phoneme_normalizer(same_phoneme)

'nghe'

# 7. Unit Normalizer

In [21]:
class UnitNormalizer(BaseBankNormalizer):
    bank_path = r'text_banks/units.json'

In [22]:
unit_normalizer = UnitNormalizer()
unit = "nm"
unit_normalizer(unit)

'na nô mét'

# 8. Number Normalizer

In [65]:
class NumberNomalizer(BaseNormalizer):

    in_regex = r"\b\d+\b"
    dot_regex = r"(\d+)\.(\d+)"
    comma_regex = r"(\d+)\,(\d+)"
    
    base_numbers_path = "text_banks/base_numbers.json"
    number_levels_path = "text_banks/number_levels.json"
      
    base_numbers = {int(key): value for key, value in load_json(base_numbers_path, is_sort_by_key=False).items()}
    number_levels = {int(key): value for key, value in load_json(number_levels_path, is_sort_by_key=False).items()}

    def _convert_number_2_digits(self, number: int):
        if number in self.base_numbers:
            return self.base_numbers[number]

        tens = number // 10
        base = number % 10
        if base > 0:
            return f"{self.base_numbers[tens]} mươi {self.base_numbers[base]}"

        return f"{self.base_numbers[tens]} mươi"

    def _convert_number_3_digits(self, number: int):
        if number == 0:
            return ""

        remainder = number % 100
        hundred = number // 100
        if remainder == 0:
            return f"{self.base_numbers[hundred]} trăm"

        if remainder < 10:
            return f"{self.base_numbers[number // 100]} trăm linh {self.base_numbers[remainder]}"

        return f"{self.base_numbers[hundred]} trăm {self._convert_number_2_digits(remainder)}"

    def number_to_vietnamese(self, number: int):
        if number == 0:
            return "không"

        if number in self.base_numbers:
            return self.base_numbers[number]

        if number < 100:
            return self._convert_number_2_digits(number)

        result = self._convert_number_3_digits(number % 1000)
        current_level = None

        for current_level in self.number_levels:
            next_level = current_level * 1000
            if number // (next_level) == 0:
                break
            level_base = number % (next_level) // current_level
            result = f"{self._convert_number_3_digits(level_base)} {self.number_levels[current_level]} {result}"

        level_base = number // current_level

        if level_base == 0:
            return result

        if level_base in self.base_numbers:
            return f"{self.base_numbers[level_base]} {self.number_levels[current_level]} {result}"

        if level_base > 99:
            return f"{self._convert_number_3_digits(level_base)} {self.number_levels[current_level]} {result}"

        if level_base > 11:
            return f"{self._convert_number_2_digits(level_base)} {self.number_levels[current_level]} {result}"

    def normalize(self, text: str) -> str:
        
        if self.out_regex is None:
            self.out_regex = lambda x: self.number_to_vietnamese(int(x.group()))
            
        text = re.sub(self.dot_regex, "", text)
        text = re.sub(self.comma_regex, lambda match: match.group(1) + " phẩy " + match.group(2), text)
            
        return super(NumberNomalizer, self).normalize(text)


In [66]:
number_normalizer = NumberNomalizer()
number = "11,2222" 
number_normalizer(number)

'mười một phẩy hai nghìn hai trăm hai mươi hai'

# 9. Text Normalizer

In [67]:
from typing import List

class TextNormalizer:
    
    def __init__(self, normalizers: List[BaseNormalizer], characters=r"[^a-zA-Z0-9\sáàảãạăắằẳẵặâấầẩẫậđéèẻẽẹêếềểễệíìỉĩịôốồổỗộơớờởỡợưứừửữựýỳỷỹỵ/.,?!]"):
        self.normalizers = normalizers
        self.characters = characters
        
    def normalize(self, text):
        text = text.lower().strip()
        text = re.sub(self.characters, "", text)
        
        for normalizer in self.normalizers:
            text = normalizer(text)

        return text
    
    def __call__(self, text, *args, **kwds):
        return self.normalize(text)

In [68]:
base_normalizers = [
    AcronymNormalizer(),
    DateNormalizer(),
    LetterNormalizer(),
    SymbolNormalizer(),
    SamePhonemeNormalizer(),
    UnitNormalizer(),
    NumberNomalizer(),
    WhitespaceNormalizer()
]

In [69]:
text_normalizer = TextNormalizer(normalizers=base_normalizers)
text = "Ngày 15/10/2024, Công ty CP ABC đã xuất kho 1.500 sản phẩm tới TP. HCM. Đơn giá mỗi sản phẩm là 250.000 VNĐ, tổng giá trị đơn hàng là 375.000.000 VNĐ. Đơn hàng này dự kiến được giao vào ngày 20/10/2024. Theo hợp đồng, công ty sẽ cung cấp thêm 2.000 sản phẩm trong quý IV. GĐ. Nguyễn Văn A đã ký duyệt vào lúc 14h30 ngày 10/10/2024."
text_normalizer(text)

'ngày mười lăm tháng mười năm hai nghìn không trăm hai mươi tư, công ty cp abc đã xuất kho sản phẩm tới thành phố. hồ chí minh. đơn giá mỗi sản phẩm là việt nam đồng, tổng giá trị đơn hàng là .không việt nam đồng. đơn hàng này dự kiến được giao vào ngày hai mươi tháng mười năm hai nghìn không trăm hai mươi tư. theo hợp đồng, công ty sẽ cung cấp thêm sản phẩm trong quý iv. giám đốc. nguyễn văn a đã ký duyệt vào lc 14h30 ngày mười tháng mười năm hai nghìn không trăm hai mươi tư.'

# 10. Text to Sequence

In [ ]:
class Text2Sequence:
    
    def to_text(self, sequence):
        pass

    def to_sequence(self, text):
        pass
    
    def __call__(self, *args, **kwds):
        pass

In [30]:
vowels = load_json("text_banks/vowels.json")

In [70]:
vowels = r"[^a-zA-Z0-9\sáàảãạăắằẳẵặâấầẩẫậđéèẻẽẹêếềểễệíìỉĩịôốồổỗộơớờởỡợưứừửữựýỳỷỹỵ.,?!]"

In [10]:
characters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ' ', '\t', '\n', '\r', 'á', 'à', 'ả', 'ã', 'ạ', 'ă', 'ắ', 'ằ', 'ẳ', 'ẵ', 'ặ', 'â', 'ấ', 'ầ', 'ẩ', 'ẫ', 'ậ', 'đ', 'é', 'è', 'ẻ', 'ẽ', 'ẹ', 'ê', 'ế', 'ề', 'ể', 'ễ', 'ệ', 'í', 'ì', 'ỉ', 'ĩ', 'ị', 'ô', 'ố', 'ồ', 'ổ', 'ỗ', 'ộ', 'ơ', 'ớ', 'ờ', 'ở', 'ỡ', 'ợ', 'ư', 'ứ', 'ừ', 'ử', 'ữ', 'ự', 'ý', 'ỳ', 'ỷ', 'ỹ', 'ỵ', '.', ',', '?', '!']

In [9]:
len(characters)

127